# Metadata and SQL filters with `Retriever.query`

This notebook shows how **chunk metadata** ends up in LanceDB and how to narrow search results when calling **`Retriever.query`**.

**Two complementary mechanisms**

1. **Server-side filter (`where`)** — Pass a Lance / DataFusion SQL predicate in `vdb_kwargs` on each `query` (or set defaults on the `Retriever`). The predicate runs inside LanceDB on table columns **`vector`**, **`text`**, **`metadata`**, **`source`**. This is implemented in `LanceDB.retrieval` as a `.where(...)` clause on the vector search.
2. **Client-side filter** — After retrieval, use `filter_hits_by_content_metadata` to keep rows whose parsed `content_metadata` matches arbitrary Python logic (see also `nemo_retriever_metadata_and_filtered_search.ipynb`).

**Metadata shape** — Ingestion stores each chunk’s `content_metadata` as a **JSON string** in the `metadata` column (compact JSON: no spaces after `:`). Sidecar columns (`meta_dataframe` / `meta_fields`) are merged into that object before upload.

**Prerequisites** — Same as the sibling metadata notebook: sample PDFs under `NEMO_RETRIEVER_ROOT/data/`, Python env with `nemo-retriever`, LanceDB, and resources for extract + embed. Use `local_ingest_embed_backend="hf"` if you are not running a local vLLM embed server.

## Imports and paths

In [1]:
import os
import shutil
from pathlib import Path

import pandas as pd

from nemo_retriever.graph_ingestor import GraphIngestor
from nemo_retriever.params import EmbedParams, ExtractParams
from nemo_retriever.retriever import Retriever
from nemo_retriever.vdb import IngestVdbOperator, filter_hits_by_content_metadata, parse_hit_content_metadata

NEMO_RETRIEVER_ROOT = Path(os.environ.get("NEMO_RETRIEVER_ROOT", "/raid/nv-ingest")).resolve()

## Configuration

Use the **same embedding model** for ingestion and for `Retriever` queries. Use a fresh LanceDB directory when you change documents or sidecar data (`overwrite=True` on upload).

In [2]:
model_name = "nvidia/llama-nemotron-embed-1b-v2"
LANCEDB_URI = str(Path("./nemo_retriever_query_filter_lancedb").resolve())
TABLE_NAME = "nv-ingest"

pdf_a = NEMO_RETRIEVER_ROOT / "data" / "woods_frost.pdf"
pdf_b = NEMO_RETRIEVER_ROOT / "data" / "multimodal_test.pdf"
files = [str(p) for p in (pdf_a, pdf_b) if p.is_file()]
if len(files) < 2:
    raise FileNotFoundError(
        f"Expected sample PDFs at {pdf_a} and {pdf_b}. Set NEMO_RETRIEVER_ROOT or copy data files."
    )

files

['/raid/nv-ingest/data/woods_frost.pdf',
 '/raid/nv-ingest/data/multimodal_test.pdf']

## Sidecar metadata

These columns are joined onto each chunk’s `content_metadata` using the document path in `source` (with `meta_join_key="auto"`).

In [3]:
meta_df = pd.DataFrame(
    {
        "source": [str(pdf_a.resolve()), str(pdf_b.resolve())],
        "meta_a": ["alpha", "bravo"],
        "meta_b": [5, 10],
    }
)
meta_path = Path("./nemo_retriever_query_filter_sidecar.csv").resolve()
meta_df.to_csv(meta_path, index=False)
meta_df

,source,meta_a,meta_b
0,/raid/nv-ingest/data/woods_frost.pdf,alpha,5
1,/raid/nv-ingest/data/multimodal_test.pdf,bravo,10


## 1) Extract and embed (graph pipeline)

In [4]:
extract_params = ExtractParams(
    extract_text=True,
    extract_tables=True,
    extract_charts=False,
    extract_images=False,
)
embed_params = EmbedParams(
    model_name=model_name,
    embed_granularity="page",
    local_ingest_embed_backend="hf",
)

ingestor = (
    GraphIngestor(run_mode="inprocess", documents=files)
    .extract(extract_params)
    .embed(embed_params)
)
result_df = ingestor.ingest()
records = result_df.to_dict("records")
len(records)

/opt/retriever_runtime/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


 -> Downloading/loading weights from HuggingFace: nvidia/nemotron-page-elements-v3
 -> Weights cached at: /root/.cache/huggingface/hub/models--nvidia--nemotron-page-elements-v3/snapshots/df62dbb631502575ac4d43b44d700b1674ab1d56/nemotron_page_elements_v3/weights.pth


install_pinned_hf_hub_download: module 'nemotron_ocr.inference.pipeline_v2' has no 'hf_hub_download' attribute; revision pinning was NOT applied.
/opt/retriever_runtime/lib/python3.12/site-packages/nemotron_ocr/inference/models/recognizer.py:90: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.tx = nn.TransformerEncoder(
/opt/retriever_runtime/lib/python3.12/site-packages/nemotron_ocr/inference/models/relational.py:105: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  nn.TransformerEncoder(
Pipeline stages:  71%|████████████████████████████████████████████████████████████████▎                         | 5/7 [00:34<00:16,  8.03s/stage, _BatchEmbedActor]/opt/retriever_runtime/lib/python3.12/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `H

5

## 2) Upload to LanceDB (sidecar merged into `metadata` JSON)

In [ ]:
if Path(LANCEDB_URI).exists():
    shutil.rmtree(LANCEDB_URI)

upload_vdb_kwargs = {
    "uri": LANCEDB_URI,
    "table_name": TABLE_NAME,
    "overwrite": True,
    "meta_dataframe": str(meta_path),
    "meta_source_field": "source",
    "meta_fields": ["meta_a", "meta_b"],
    "meta_join_key": "auto",
}

uploader = IngestVdbOperator(vdb_op="lancedb", vdb_kwargs=upload_vdb_kwargs)
uploader.process(records)

## 3) Inspect metadata on hits from `Retriever.query`

Each hit includes a **`metadata`** string field (JSON). Parse it with `parse_hit_content_metadata` to work with **`meta_a`**, **`meta_b`**, page numbers, etc.

In [6]:
base_vdb = {"uri": LANCEDB_URI, "table_name": TABLE_NAME}

retriever = Retriever(
    vdb="lancedb",
    vdb_kwargs=base_vdb,
    embedder=model_name,
    top_k=8,
    local_query_embed_backend="hf",
)

q = "introduction summary table"
hits = retriever.query(q, top_k=8)
print("num hits:", len(hits))
if hits:
    first = hits[0]
    cm = parse_hit_content_metadata(first)
    print("parsed content_metadata keys (sample):", sorted(cm.keys())[:20])
    print("meta_a:", cm.get("meta_a"), "meta_b:", cm.get("meta_b"))
    print("text preview:", (first.get("text") or "")[:200])

num hits: 5
parsed content_metadata keys (sample): ['meta_a', 'meta_b', 'page_number']
meta_a: bravo meta_b: 10
text preview: TestingDocument
A sample document with headings and placeholder text
Introduction
This is a placeholder document that can be used for any purpose. It contains some 
headings and some placeholder t


## 4) Server-side filter: `query(..., vdb_kwargs={..., "where": "..."})`

Per-call **`vdb_kwargs`** are merged with the retriever’s defaults. Pass a SQL predicate as **`where`** (alias **`_filter`**) to restrict rows **before** `top_k` is applied in the database.

**Examples**

- Filter on **`text`**: simple equality or `LIKE`.
- Filter on **`metadata`**: the column is a string; for JSON substrings use `LIKE` with a fragment that matches **compact** JSON (e.g. `"meta_a":"alpha"`).

Escape single quotes in SQL strings by doubling them (`''`).

In [7]:
# Restrict to chunks whose serialized content_metadata contains meta_a == "alpha"
# (matches ingestion JSON like {"meta_a":"alpha",...})
pred_alpha = "metadata LIKE '%\"meta_a\":\"alpha\"%'"

hits_alpha = retriever.query(q, top_k=8, vdb_kwargs={**base_vdb, "where": pred_alpha})
print("hits with server-side meta_a filter:", len(hits_alpha))
for h in hits_alpha[:3]:
    print(parse_hit_content_metadata(h).get("meta_a"), (h.get("text") or "")[:100])

hits with server-side meta_a filter: 2
alpha # Collection Year 1 A Boy's Will 1913 2 North of Boston 1914 3 Mountain Interval 1916 4 New Hampshir
alpha Stopping by Woods on a Snowy Evening, By Robert Frost
Figure 1: Snowy Woods
Whose woods these are 


In [8]:
# Same predicate using the _filter alias (useful for Milvus-style call sites)
hits_alias = retriever.query(q, top_k=8, vdb_kwargs={**base_vdb, "_filter": pred_alpha})
assert len(hits_alias) == len(hits_alpha)

## 5) Compare: server-side `where` vs client-side `filter_hits_by_content_metadata`

- **`where`** reduces work in LanceDB and can shrink the candidate set before vector ranking (depending on planner behavior).
- **`filter_hits_by_content_metadata`** is flexible (any Python predicate) but runs after the vector search returns.

In [9]:
hits_wide = retriever.query(q, top_k=16)
client_only = filter_hits_by_content_metadata(
    hits_wide, lambda m: m.get("meta_a") == "bravo" and m.get("meta_b", 0) >= 10
)
print("wide search then client filter:", len(client_only))

# Approximate the same intent in SQL on the JSON string (meta_b is stored as a number in JSON)
pred_bravo = "metadata LIKE '%\"meta_a\":\"bravo\"%' AND metadata LIKE '%\"meta_b\":10%'"
hits_sql = retriever.query(q, top_k=16, vdb_kwargs={**base_vdb, "where": pred_bravo})
print("single query with where:", len(hits_sql))

wide search then client filter: 3
single query with where: 3
